# A score, then a sign

An SVM starts with one practical question. You have two groups. For a new point, which group do you call?

The machine does not jump straight to a label. It first computes a **score**. In the plane, a score is a weighted sum plus a shift:

$$
s = w_1 x_1 + w_2 x_2 + b = \mathbf{w}\cdot\mathbf{x} + b
$$

The label is only the sign of that score:

$$
\hat y =
\begin{cases}
+1 & \text{if } s > 0 \\
-1 & \text{if } s < 0
\end{cases}
$$

If $s = 0$, the point sits on the line itself. The rule has nothing to say. A careful program reports "on the boundary" instead of inventing a class.


## Worked scores

Take $\mathbf{w} = (1, 0)$ and $b = -1$. The score collapses to one coordinate:

$$
s = x_1 - 1
$$

The line $s = 0$ is the vertical line $x_1 = 1$. Everything to the right is called positive. Everything to the left is called negative.

| Point | Score $x_1 - 1$ | Call |
|---|---:|---|
| $(2, 0)$ | $1$ | $+1$ |
| $(0, 1)$ | $-1$ | $-1$ |
| $(0, -1)$ | $-1$ | $-1$ |
| $(4, 0)$ | $3$ | $+1$ |
| $(1, 5)$ | $0$ | on the line |

The score $3$ is larger than the score $1$. That means the point is farther from the line. It does **not** mean the point is three times as likely. A score is a signed distance in disguise, not a probability.


In [1]:
# Recompute every score in the table, then apply the sign rule.
points = {
    "(2, 0)": (2, 0),
    "(0, 1)": (0, 1),
    "(0, -1)": (0, -1),
    "(4, 0)": (4, 0),
    "(1, 5)": (1, 5),
}
hand = {
    "(2, 0)": 1,
    "(0, 1)": -1,
    "(0, -1)": -1,
    "(4, 0)": 3,
    "(1, 5)": 0,
}

def score(point):
    x1, x2 = point
    return 1 * x1 + 0 * x2 + (-1)

def call(value):
    if value > 0:
        return "+1"
    if value < 0:
        return "-1"
    return "on the boundary"

for name, point in points.items():
    value = score(point)
    assert value == hand[name]
    print(name, "score", value, "call", call(value))


(2, 0) score 1 call +1
(0, 1) score -1 call -1
(0, -1) score -1 call -1
(4, 0) score 3 call +1
(1, 5) score 0 call on the boundary


## One change: flip the ruler

Replace $(\mathbf{w}, b)$ by $(-\mathbf{w}, -b)$. Every score changes sign, so every call flips.

$$
(-\mathbf{w})\cdot\mathbf{x} + (-b) = -(\mathbf{w}\cdot\mathbf{x} + b)
$$

For the same line, $(2, 0)$ now scores $-1$ and is called negative. The geometry of the line did not move. Only the names of the two sides moved. When you fit an SVM you therefore have to keep the labels attached to the correct side.


In [2]:
# Flipping (w, b) flips every call, including the point that was on the line.
def signed_call(point, w1, w2, b):
    value = w1 * point[0] + w2 * point[1] + b
    if value > 0:
        return "+1"
    if value < 0:
        return "-1"
    return "on the boundary"

point = (2, 0)
assert signed_call(point, 1, 0, -1) == "+1"
assert signed_call(point, -1, 0, 1) == "-1"
assert signed_call((1, 5), -1, 0, 1) == "on the boundary"
print("original call", signed_call(point, 1, 0, -1))
print("flipped call", signed_call(point, -1, 0, 1))


original call +1
flipped call -1


## A pitfall

Do not turn the score into a class with a cutoff other than zero unless you have a reason. The later lessons fix the scale of $\mathbf{w}$ and $b$ so that the two margin edges lie at $+1$ and $-1$. If you then classify at $0.5$ instead of $0$, you have moved the line.

## What to carry forward

Prediction is $\operatorname{sign}(\mathbf{w}\cdot\mathbf{x}+b)$. The size of the score says how far the point is from the line. It is not a probability.
